# Kadanoff-Baym Equations with α-Relaxation**Goal:** Apply α-transform relaxation to the time-dependent non-equilibrium Dyson equation on the Keldysh contour.The Kadanoff-Baym equations are the non-equilibrium generalisation of the Dyson equation:$$G^R(t,t') = G_0^R(t,t') + \int dt_1\, G_0^R(t,t_1)\, \Sigma^R(t_1, t')\, G^R(t_1, t')$$$$G^<(t,t') = \int dt_1 dt_2\, G^R(t,t_1)\, \Sigma^<(t_1,t_2)\, G^A(t_2,t')$$The propagators split into three Keldysh components:- **Retarded** $G^R$: causal response- **Advanced** $G^A = [G^R]^\dagger$: anti-causal- **Lesser** $G^<$: occupation / distribution**Key question:** Can α-relaxation stabilize the time-stepping without damping physical signals?**Predictions from earlier results:**- The piecewise α(ω) result (α*≈0.5 everywhere) suggests a single global α should suffice- But each Keldysh component may need a different α (retarded vs lesser have different stability properties)- The Lyapunov Z₂ symmetry Λ(α) = Λ(−α) should manifest as equivalent stabilization for ±α

In [ ]:
import syssys.path.insert(0, '..')import numpy as npimport matplotlib.pyplot as pltfrom petrification.dyson import (    scalar_dyson_iterate, scalar_dyson_exact,    scalar_dyson_stability, spectral_dyson_scan,    quadratic_self_energy, hubbard_atom_self_energy)plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12})

## §2. Time-Domain Kadanoff-Baym SolverDiscretize the KB equations on a time grid. The retarded component satisfies a Volterra integral equation that we solve by stepping forward in time. At each step, the self-energy depends on $G$ itself (self-consistency), making this a fixed-point iteration at each time step.

In [ ]:
# ─── Discretized KB equations for a single-level model ───def kb_retarded_step(G_R, Sigma_R, h0, dt, t_idx, alpha=1.0):    """    Advance G^R(t_idx, 0:t_idx) by one time step using the Volterra equation:    i∂_t G^R(t,t') = δ(t-t') + h₀ G^R(t,t') + ∫ Σ^R(t,t₁) G^R(t₁,t') dt₁    With alpha-relaxation on the self-consistent loop.    """    n = t_idx    # G^R(t,t) = -i (initial condition from equal-time anti-commutator)    G_R[n, n] = -1j    # For t' < t: advance using trapezoidal rule on the memory integral    for tp in range(n):        # Memory integral: ∫₀^t Σ^R(t,t₁) G^R(t₁,t') dt₁        mem = 0.0        for t1 in range(tp, n):            w = dt  # trapezoidal weight            if t1 == tp or t1 == n-1:                w *= 0.5            mem += w * Sigma_R[n, t1] * G_R[t1, tp]        # Euler step: G^R(n+1, tp) ≈ G^R(n, tp) - i dt (h₀ G^R(n,tp) + mem)        G_new = G_R[n-1, tp] - 1j * dt * (h0 * G_R[n-1, tp] + mem)        # Alpha-relaxation: mix new with old        if n > 1 and tp < n-1:            G_R[n, tp] = alpha * G_new + (1 - alpha) * G_R[n-1, tp]        else:            G_R[n, tp] = G_new    return G_Rdef kb_lesser_step(G_less, G_R, Sigma_less, dt, t_idx, alpha=1.0):    """    Advance G^<(t_idx, 0:t_idx) using:    G^<(t,t') = ∫∫ G^R(t,t₁) Σ^<(t₁,t₂) G^A(t₂,t') dt₁ dt₂    where G^A(t₂,t') = [G^R(t',t₂)]^*    """    n = t_idx    for tp in range(n+1):        # Double integral (simplified for computational feasibility)        val = 0.0        for t1 in range(n+1):            for t2 in range(tp+1):                w = dt * dt                if t1 == 0 or t1 == n:                    w *= 0.5                if t2 == 0 or t2 == tp:                    w *= 0.5                G_A_t2_tp = np.conj(G_R[tp, t2]) if tp >= t2 else np.conj(G_R[t2, tp])                val += w * G_R[n, t1] * Sigma_less[t1, t2] * G_A_t2_tp        G_new = val        if n > 0 and tp < n:            old = G_less[n-1, tp] if n > 0 else 0.0            G_less[n, tp] = alpha * G_new + (1 - alpha) * old        else:            G_less[n, tp] = G_new    return G_lessdef solve_kb_equations(h0, U, beta, n_times, t_max, alpha_R=1.0, alpha_less=1.0):    """    Solve KB equations for a Hubbard atom model.    h0: bare energy    U: interaction strength    beta: inverse temperature (for initial equilibrium)    alpha_R: relaxation parameter for retarded component    alpha_less: relaxation parameter for lesser component    """    dt = t_max / n_times    times = np.linspace(0, t_max, n_times)    # Allocate Green's functions    G_R = np.zeros((n_times, n_times), dtype=complex)    G_less = np.zeros((n_times, n_times), dtype=complex)    # Self-energies (2nd Born: Σ ∝ U² G)    Sigma_R = np.zeros((n_times, n_times), dtype=complex)    Sigma_less = np.zeros((n_times, n_times), dtype=complex)    # Initial equilibrium self-energy (constant)    nf = 1.0 / (np.exp(beta * h0) + 1)  # Fermi function    # Initialize: free retarded propagator G₀^R(t,t') = -i θ(t-t') e^{-ih₀(t-t')}    for i in range(n_times):        for j in range(i+1):            G_R[i, j] = -1j * np.exp(-1j * h0 * (times[i] - times[j]))            G_less[i, j] = 1j * nf * np.exp(-1j * h0 * (times[i] - times[j]))    # Self-consistent loop (simplified: a few iterations)    n_sc_iter = 5    residuals = []    for sc_iter in range(n_sc_iter):        G_R_old = G_R.copy()        # Update self-energy from current G (2nd Born approximation)        for i in range(n_times):            for j in range(i+1):                Sigma_R[i, j] = U**2 * G_R[i, j] * abs(G_less[i, j])**2                Sigma_less[i, j] = U**2 * G_less[i, j] * abs(G_R[i, j])**2        # Solve KB equations time-step by time-step        for t_idx in range(1, n_times):            G_R = kb_retarded_step(G_R, Sigma_R, h0, dt, t_idx, alpha=alpha_R)            G_less = kb_lesser_step(G_less, G_R, Sigma_less, dt, t_idx, alpha=alpha_less)        # Residual        res = np.max(np.abs(G_R - G_R_old))        residuals.append(res)    return times, G_R, G_less, residualsprint('KB solver defined.')

## §3. α-Relaxation Impact on KB ConvergenceCompare convergence with α = 1 (naive), α = 0.5, and α = 0.3 for a strongly interacting system.

In [ ]:
# Small grid for tractabilityn_times = 40t_max = 5.0h0 = 0.0U = 2.0  # strong couplingbeta = 5.0alpha_configs = [    (1.0, 1.0, 'Naive (α=1)'),    (0.5, 0.5, 'α=0.5 (both)'),    (0.3, 0.3, 'α=0.3 (both)'),    (0.5, 0.3, 'α_R=0.5, α_<=0.3'),    (-0.5, -0.5, 'α=−0.5 (Z₂ test)'),]fig, axes = plt.subplots(1, 2, figsize=(14, 5))results = {}for alpha_R, alpha_less, label in alpha_configs:    times, G_R, G_less, residuals = solve_kb_equations(        h0, U, beta, n_times, t_max, alpha_R=alpha_R, alpha_less=alpha_less    )    results[label] = (times, G_R, G_less, residuals)    axes[0].semilogy(range(len(residuals)), residuals, 'o-', markersize=4, label=label)axes[0].set(xlabel='Self-consistent iteration', ylabel='Max |ΔG^R|',            title=f'KB Self-Consistency Convergence (U={U}, β={beta})')axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)# Compare spectral functions (diagonal G^R)for label, (times, G_R, G_less, _) in results.items():    GR_t = G_R[:, 0]    axes[1].plot(times, -np.imag(GR_t), linewidth=1.5, label=label)axes[1].set(xlabel='Time t', ylabel='-Im G^R(t, 0)',            title='Retarded Propagator')axes[1].legend(fontsize=9)axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## §4. Frequency-Domain ValidationCross-check: solve the equilibrium Dyson equation in frequency domain (using existing `spectral_dyson_scan`) and compare spectral functions.

In [ ]:
# Frequency-domain referenceeta_broadening = 0.05omega_grid = np.linspace(-6, 6, 300) + 1j * eta_broadeningsigma_func = quadratic_self_energy(U)alpha_freq_configs = [1.0, 0.5, 0.3]fig, axes = plt.subplots(1, 2, figsize=(14, 5))for alpha in alpha_freq_configs:    G_conv, n_conv, iters = spectral_dyson_scan(omega_grid, h0, sigma_func,                                                 alpha=alpha, n_iter=500, tol=1e-12)    A_omega = -np.imag(G_conv) / np.pi  # spectral function    axes[0].plot(np.real(omega_grid), A_omega, linewidth=1.5,                 label=f'α={alpha} ({n_conv}/{len(omega_grid)} conv.)')    axes[1].plot(np.real(omega_grid), iters, linewidth=1, label=f'α={alpha}')# Exact referenceG_exact = np.array([scalar_dyson_exact(w, h0, U) for w in omega_grid])A_exact = -np.imag(G_exact) / np.piaxes[0].plot(np.real(omega_grid), A_exact, 'k--', linewidth=1, label='Exact', alpha=0.5)axes[0].set(xlabel='ω', ylabel='A(ω)', title=f'Spectral Function (U={U}, ε₀={h0})')axes[0].legend(fontsize=9)axes[0].grid(True, alpha=0.3)axes[1].set(xlabel='ω', ylabel='Iterations to converge',            title='Convergence Speed vs Frequency')axes[1].legend(fontsize=9)axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## §5. Component-Dependent α: Do Retarded and Lesser Need Different Mixing?The piecewise α(ω) investigation showed α*≈0.5 everywhere in frequency. But in time-domain KB, the retarded and lesser components have fundamentally different analytic structures. Test whether they benefit from different α values.

In [ ]:
# Sweep α_R and α_< independentlyalpha_R_values = np.linspace(0.1, 0.9, 9)alpha_less_values = np.linspace(0.1, 0.9, 9)convergence_map = np.zeros((len(alpha_R_values), len(alpha_less_values)))stability_map = np.zeros((len(alpha_R_values), len(alpha_less_values)))for i, aR in enumerate(alpha_R_values):    for j, aL in enumerate(alpha_less_values):        _, G_R, G_less, residuals = solve_kb_equations(            h0, U, beta, n_times=30, t_max=4.0, alpha_R=aR, alpha_less=aL        )        convergence_map[i, j] = residuals[-1] if len(residuals) > 0 else 1e10        # Stability: check if G^R stays bounded        max_G = np.max(np.abs(G_R))        stability_map[i, j] = max_Gfig, axes = plt.subplots(1, 2, figsize=(14, 5))im1 = axes[0].pcolormesh(alpha_less_values, alpha_R_values,                          np.log10(convergence_map + 1e-16),                          cmap='viridis_r', shading='auto')axes[0].set(xlabel='α_lesser', ylabel='α_retarded',            title='log₁₀(Final SC Residual)')plt.colorbar(im1, ax=axes[0])im2 = axes[1].pcolormesh(alpha_less_values, alpha_R_values,                          np.log10(stability_map + 1e-16),                          cmap='RdYlGn_r', shading='auto')axes[1].set(xlabel='α_lesser', ylabel='α_retarded',            title='log₁₀(max |G^R|) — Stability')plt.colorbar(im2, ax=axes[1])# Mark optimalopt_idx = np.unravel_index(np.argmin(convergence_map), convergence_map.shape)axes[0].plot(alpha_less_values[opt_idx[1]], alpha_R_values[opt_idx[0]], 'r*', markersize=15)print(f'Optimal: α_R = {alpha_R_values[opt_idx[0]]:.2f}, α_< = {alpha_less_values[opt_idx[1]]:.2f}')print(f'Residual: {convergence_map[opt_idx]:.2e}')plt.tight_layout()plt.show()

## §6. Z₂ Symmetry in KB EquationsTest whether α and −α give identical results in the Kadanoff-Baym context. From the Lyapunov analysis, we found Λ(α) = Λ(−α) for the logistic map. Does this extend to the KB self-consistency?

In [ ]:
# Z₂ test: compare α = +0.5 vs α = -0.5 in frequency domain (cleaner)omega_z2 = np.linspace(-6, 6, 200) + 1j * eta_broadeningfor U_test in [1.0, 2.0, 3.0]:    sigma_z2 = quadratic_self_energy(U_test)    G_pos, n_pos, it_pos = spectral_dyson_scan(omega_z2, h0, sigma_z2, alpha=0.5, n_iter=500)    G_neg, n_neg, it_neg = spectral_dyson_scan(omega_z2, h0, sigma_z2, alpha=-0.5, n_iter=500)    diff = np.max(np.abs(G_pos - G_neg))    print(f'U = {U_test}: max |G(α=+0.5) - G(α=-0.5)| = {diff:.2e}, '          f'conv: +0.5={n_pos}/{len(omega_z2)}, -0.5={n_neg}/{len(omega_z2)}')fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Detailed comparison for U=2sigma_z2 = quadratic_self_energy(2.0)G_pos, _, it_pos = spectral_dyson_scan(omega_z2, h0, sigma_z2, alpha=0.5, n_iter=500)G_neg, _, it_neg = spectral_dyson_scan(omega_z2, h0, sigma_z2, alpha=-0.5, n_iter=500)axes[0].plot(np.real(omega_z2), -np.imag(G_pos)/np.pi, 'b-', linewidth=2, label='α = +0.5')axes[0].plot(np.real(omega_z2), -np.imag(G_neg)/np.pi, 'r--', linewidth=2, label='α = -0.5')axes[0].set(xlabel='ω', ylabel='A(ω)', title='Spectral Function: Z₂ test')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(np.real(omega_z2), it_pos, 'b-', linewidth=1, label='α = +0.5')axes[1].plot(np.real(omega_z2), it_neg, 'r--', linewidth=1, label='α = -0.5')axes[1].set(xlabel='ω', ylabel='Iterations', title='Convergence Speed: Z₂ test')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('Z₂ Symmetry: Λ(α) = Λ(−α) in Dyson Equation', fontsize=14)plt.tight_layout()plt.show()

## §7. Quench Dynamics: Sudden Interaction Turn-OnThe canonical KB benchmark: start in equilibrium at $U = 0$, suddenly turn on interaction at $t = 0$. Track how the spectral weight redistributes. This is where naive time-stepping typically becomes unstable for strong $U$.

In [ ]:
# Quench protocol: U(t) = 0 for t < 0, U_final for t >= 0U_quench_values = [1.0, 2.0, 3.0]n_times_quench = 50t_max_quench = 8.0fig, axes = plt.subplots(len(U_quench_values), 2, figsize=(14, 4*len(U_quench_values)))for row, U_q in enumerate(U_quench_values):    for alpha_val, color, lstyle in [(1.0, 'red', '-'), (0.5, 'blue', '-'), (0.3, 'green', '-')]:        times, G_R, G_less, residuals = solve_kb_equations(            h0, U_q, beta, n_times_quench, t_max_quench,            alpha_R=alpha_val, alpha_less=alpha_val        )        # Observable: occupation n(t) = -i G^<(t,t)        n_t = -1j * np.diag(G_less)        axes[row, 0].plot(times, np.real(n_t), color=color, linestyle=lstyle,                          linewidth=1.5, label=f'α={alpha_val}')        # Spectral weight: Im G^R(t, 0)        axes[row, 1].plot(times, -np.imag(G_R[:, 0]), color=color, linestyle=lstyle,                          linewidth=1.5, label=f'α={alpha_val}')    axes[row, 0].set(ylabel=f'U={U_q}\nn(t)', title='Occupation' if row == 0 else '')    axes[row, 0].legend(fontsize=8)    axes[row, 0].grid(True, alpha=0.3)    axes[row, 1].set(title='-Im G^R(t, 0)' if row == 0 else '')    axes[row, 1].legend(fontsize=8)    axes[row, 1].grid(True, alpha=0.3)axes[-1, 0].set_xlabel('Time')axes[-1, 1].set_xlabel('Time')plt.suptitle('Interaction Quench: U(t<0)=0 → U(t≥0)=U_final', fontsize=14)plt.tight_layout()plt.show()

## §8. Energy Conservation CheckA good KB solver conserves total energy. Does α-relaxation introduce spurious energy drift, or is it purely a solver-level improvement that preserves conservation laws?

In [ ]:
# Compare energy conservation across alpha valuesU_test = 2.0n_t_test = 60t_max_test = 10.0energies = {}for alpha_val in [1.0, 0.5, 0.3]:    times, G_R, G_less, _ = solve_kb_equations(        h0, U_test, beta, n_t_test, t_max_test,        alpha_R=alpha_val, alpha_less=alpha_val    )    dt = times[1] - times[0]    # Kinetic energy: h₀ * n(t)    n_t = -1j * np.diag(G_less)    E_kin = h0 * np.real(n_t)    # Interaction energy (simplified): ½ U² ∫ Σ^R G^< dt'    E_int = np.zeros(len(times))    for i in range(len(times)):        contrib = 0.0        for j in range(i+1):            Sigma_R_ij = U_test**2 * G_R[i, j] * abs(G_less[i, j])**2            contrib += dt * np.real(Sigma_R_ij * G_less[j, i].conj())        E_int[i] = 0.5 * contrib    E_total = E_kin + E_int    energies[alpha_val] = E_totalfig, ax = plt.subplots(figsize=(10, 6))for alpha_val, E in energies.items():    dE = E - E[0]    ax.plot(times, dE, linewidth=2, label=f'α = {alpha_val}')ax.set(xlabel='Time', ylabel='E(t) - E(0)',       title=f'Energy Conservation Check (U={U_test})')ax.legend()ax.grid(True, alpha=0.3)ax.axhline(0, color='k', linestyle='--', alpha=0.3)plt.tight_layout()plt.show()for alpha_val, E in energies.items():    drift = abs(E[-1] - E[0])    max_dev = np.max(np.abs(E - E[0]))    print(f'α = {alpha_val}: energy drift = {drift:.4e}, max deviation = {max_dev:.4e}')

## §9. Assessment| Result | Status | Notes ||--------|--------|-------|| α-relaxation on KB equations | Fill after running | Does it improve convergence? || Global α≈0.5 sufficiency | Fill after running | Predicted from piecewise result || Component-dependent α | Fill after running | Do R and < need different α? || Z₂ symmetry in Dyson | Fill after running | Does Λ(α)=Λ(−α) hold? || Quench dynamics stability | Fill after running | Key benchmark || Energy conservation | Fill after running | Does α introduce artifacts? |### Key insight for the α-transform theoryThe KB equations test whether α-relaxation is fundamentally a **solver trick** (accelerating convergence of a fixed-point iteration) or a **physical mixing** (modifying the effective dynamics). If energy conservation is preserved at all α values, it's purely a solver improvement. If different α values lead to different conserved quantities, the interpretation is deeper.### Hints for higher dimensions- **Matrix-valued G → matrix α:** In multi-orbital KB equations, α becomes a matrix mixing different orbital channels. The optimal α would be related to the spectral radius of the self-energy.- **Spatial lattice:** For lattice KB equations with k-space, α(k,ω) could exploit structure in the Brillouin zone.- **Keldysh contour deformation:** The α-transform could be interpreted as a contour deformation in the complex time plane, connecting to analyticity questions.